In [10]:
from pathlib import Path
import pandas as pd
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from typing import Literal
from sklearn.preprocessing import MinMaxScaler,StandardScaler

data_path = Path("../data/processed/NVDA_outliers.csv")
df = pd.read_csv(data_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               1215 non-null   object 
 1   adj close          1215 non-null   float64
 2   close              1215 non-null   float64
 3   high               1215 non-null   float64
 4   low                1215 non-null   float64
 5   open               1215 non-null   float64
 6   volume             1215 non-null   int64  
 7   ticker             1215 non-null   object 
 8   daily_rtn          1215 non-null   float64
 9   excess_return      1215 non-null   float64
 10  outlier_iqr        1215 non-null   bool   
 11  outlier_winsorize  1215 non-null   float64
 12  outlier_z          1215 non-null   bool   
dtypes: bool(2), float64(8), int64(1), object(2)
memory usage: 106.9+ KB


In [11]:
#cumulative return over the last five trading days
df["mom_5"] = (df["adj close"]/df["adj close"].shift(5)-1).shift(1)
df["log_return"] = np.log(df["adj close"]).diff().shift(1)
df["vol_std_5"] = df["log_return"].rolling(5).std().shift(1)
df['next_excess_return'] = df['excess_return'].shift(-1) 
df = df.copy().dropna().reset_index(drop=True)
df["date"] = pd.to_datetime(df["date"])
df.to_parquet("../data/processed/NVDA_feature_engineering.parquet",index=False)
df

,date,adj close,close,high,low,open,volume,ticker,daily_rtn,excess_return,outlier_iqr,outlier_winsorize,outlier_z,mom_5,log_return,vol_std_5,next_excess_return
0,2020-08-31,13.330562,13.374500,13.575000,13.037750,13.182750,500840000,NVDA,0.017246,0.019441,False,0.019441,False,0.036602,0.040314,0.020906,0.026168
1,2020-09-01,13.779715,13.821000,13.993750,13.436500,13.480000,511316000,NVDA,0.033693,0.026168,False,0.026168,False,0.051434,0.017099,0.019453,0.022656
2,2020-09-02,14.303645,14.346500,14.726750,13.900000,14.703750,874012000,NVDA,0.038022,0.022656,False,0.022656,False,0.084324,0.033138,0.019705,-0.057649
3,2020-09-03,12.976623,13.015500,13.884500,12.878750,13.828750,945128000,NVDA,-0.092775,-0.057649,False,-0.035831,False,0.123525,0.037317,0.021437,-0.022062
4,2020-09-04,12.584794,12.622500,13.175000,11.704750,12.783750,1463684000,NVDA,-0.030195,-0.022062,False,-0.022062,False,0.030974,-0.097365,0.021361,-0.028453
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202,2025-08-11,182.059998,182.059998,183.839996,180.250000,182.050003,138323200,NVDA,-0.003503,-0.000999,False,-0.000999,False,0.051692,0.010620,0.022129,-0.005303
1203,2025-08-12,183.160004,183.160004,184.479996,179.460007,182.960007,145485700,NVDA,0.006042,-0.005303,False,-0.005303,False,0.011444,-0.003509,0.016267,-0.011802
1204,2025-08-13,181.589996,181.589996,183.970001,179.350006,182.619995,179871700,NVDA,-0.008572,-0.011802,False,-0.011802,False,0.027488,0.006024,0.008541,0.002065
1205,2025-08-14,182.020004,182.020004,183.020004,179.460007,179.750000,129554000,NVDA,0.002368,0.002065,False,0.002065,False,0.012095,-0.008609,0.005305,-0.005728
